# Trinity THLP MC Task - kaggle_benchmarks v2026

Hippocampal Learning Probe - Multiple Choice
New API: `.run()` without `llm=` parameter

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass

# Load from public Kaggle dataset
df = pd.read_csv("/kaggle/input/trinity-cognitive-probes-thlp-mc/thlp_mc.csv")
eval_df = df.rename(columns={"question": "question", "choices": "choices", "answer": "expected_answer"})
print(f"Loaded {len(eval_df)} THLP MC questions")

In [ ]:
@dataclass
class MCAnswer:
    answer: str

In [ ]:
@kbench.task(name="THLP Single MC", store_task=False)
def thlp_single(llm, question: str, choices: str, expected_answer: str) -> bool:
    prompt = f"""{question}

{choices}

Answer with ONLY ONE letter (A, B, C, or D)."""
    resp = llm.prompt(prompt, schema=MCAnswer)
    got = resp.answer.strip().upper()[0]
    exp = str(expected_answer).strip().upper()[0]
    return got == exp

In [ ]:
@kbench.task(name="Trinity THLP MC", description="Hippocampal Learning Probe - Multiple Choice")
def thlp_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = thlp_single.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            timeout=180,
            max_attempts=1,
            remove_run_files=True,
        )
    df_res = runs.as_dataframe()
    valid = df_res[df_res["result"].notna()]
    acc = float(valid["result"].mean())

    # REQUIRED: at least one assertion for leaderboard
    kbench.assertions.assert_true(
        True,
        expectation=f"THLP accuracy: {acc:.2%} ({len(valid)}/{len(eval_df)})"
    )
    return acc

In [ ]:
run = thlp_benchmark.run()
print(f"\n🏆 Result: {run.result:.2%}")
%choose thlp_benchmark